# JUICE RPWI HF HK: QL -- 2026/3/13

In [ ]:
import datetime
import matplotlib.pyplot as plt
import numpy as np
import sys

sys.path.append('./lib/')
import juice_hk_read as juice_hk

In [ ]:
dump_mode = 1

# Setting and Read CDF file: set by User

In [ ]:
# date='20240125'   # date='20240126'
# date='20240701'   # date='20240702'   # date='20240706'
# date='20240819'   # date='20240820'   # date='20240821'   # date='20240822'   # date='20240823'   # date='20240909'
date='20260223'
#date='20260224'
#date='20260225'
#date='20260226'
#date='20260227'
#date='20260228'

# Ground
# date='HK'         # linked to /Users/user/0-python/JUICE_data/HK/HK/

In [ ]:
# *** Directory set: set by User ***
work_dir = './ql/'              # Plot dump folder
if date[0:2] == '20':
    base_dir = '/Users/D-Univ/data/data-JUICE/datasets/'
elif date[0:2] == 'HK':
    base_dir = '/Users/user/0-python/JUICE_data/HK/'            # Ground test   [with Link revision]
print(base_dir)

# (1) read CDF files

In [ ]:
# HF: 32
data_hf, err_hf         = juice_hk.juice_readhk(date, 'LWYHK00032', base_dir=base_dir)
if err_hf == 0: 
    print("==> read HF-HK ASW1");           hk_hf = juice_hk.juice_gethk_hf(data_hf, 0)
else:
    data_hf, err_hf     = juice_hk.juice_readhk(date, 'LWYHK10033', base_dir=base_dir)
    if err_hf == 0:
        print("==> read HF-HK ASW2");       hk_hf = juice_hk.juice_gethk_hf(data_hf, 1)
    else: 
        data_hf, err_hf = juice_hk.juice_readhk(date, 'LWYHK20033', base_dir=base_dir)
        if err_hf == 0:
            print("==> read HF-HK ASW3");   hk_hf = juice_hk.juice_gethk_hf(data_hf, 1)

In [ ]:
# DPU: 64
data_dpu, err_dpu = juice_hk.juice_readhk(date, 'LWYHK00064', base_dir=base_dir)
if err_dpu == 0:
    print("==> read DPU-HK ASW1");          hk_dpu = juice_hk.juice_gethk_dpu(data_dpu, 0)
else: 
    data_dpu, err_dpu = juice_hk.juice_readhk(date, 'LWYHK10064', base_dir=base_dir)
    if err_dpu == 0:
        print("==> read DPU-HK ASW2/3");    hk_dpu = juice_hk.juice_gethk_dpu(data_dpu, 1)
    else: 
        print("==> no DPU-HK")

In [ ]:
# LVPS: 80
data_lvps, err_lvps = juice_hk.juice_readhk(date, 'LWYHK00080', base_dir=base_dir)
if err_lvps == 0:
    print("==> read LVPS-HK")
    hk_lvps = juice_hk.juice_gethk_lvps(data_lvps)
else: 
    print("==> no LVPS HK")

In [ ]:
if err_hf == 0:
    temp_lim = [-150.0, 100.0]
    temp_rwi_ch1 = np.where((hk_hf.temp_rwi_ch1 < temp_lim[0]) |
                          (hk_hf.temp_rwi_ch1 > temp_lim[1]), np.nan, hk_hf.temp_rwi_ch1)
    temp_rwi_ch2 = np.where((hk_hf.temp_rwi_ch2 < temp_lim[0]) |
                          (hk_hf.temp_rwi_ch2 > temp_lim[1]), np.nan, hk_hf.temp_rwi_ch2)
    temp_hf_fpga = np.where((hk_hf.temp_hf_fpga < temp_lim[0]) |
                            (hk_hf.temp_hf_fpga > temp_lim[1]), np.nan, hk_hf.temp_hf_fpga)

# (2) Plot HK values

In [ ]:
title_label = 'JUICE/RPWI HF HK ' + date[0:4] + '-' + date[4:6] + '-' + date[6:8]
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, figsize=(14, 11), sharex=True)

if err_hf == 0:
    ax1.plot(hk_hf.epoch, temp_hf_fpga, '.c', markersize=6, linewidth=0.5, label='HF(FPGA)')
if err_dpu == 0:
    ax1.plot(hk_dpu.epoch, hk_dpu.hf_temp,    '.k', markersize=5, label='HF(PCB)')
    ax1.plot(hk_dpu.epoch, hk_dpu.lvps_temp,  '--b', markersize=1, label='LVPS(PCB)')
ax1.set_ylabel('HF T [degC]');  ax1.legend();  ax1.set_title(title_label)

if err_hf == 0:
    ax2.plot(hk_hf.epoch, temp_rwi_ch1, '.c', markersize=6, linewidth=0.5, label='RWI CH1')
    ax2.plot(hk_hf.epoch, temp_rwi_ch2, '.y', markersize=6, linewidth=0.5, label='RWI CH2')
if err_dpu == 0:
    ax2.plot(hk_dpu.epoch, hk_dpu.scm_temp,  '.b', markersize=1, label='SCM')
ax2.set_ylabel('RWI T [degC]');  ax2.legend()

if err_hf == 0:
    ax3.plot(hk_hf.epoch,   hk_hf.rwi_on,       '-c', markersize=6, linewidth=0.5, label='RWI ON (1:U + 2:V + 4:W)')
if err_lvps == 0:
    ax3.plot(hk_lvps.epoch, hk_lvps.vol_hf_85,  '.g', markersize=3, label='RWI 8.5V (LVPS)')
    ax3.plot(hk_lvps.epoch, hk_lvps.vol_hf_33,  '.k', markersize=3, label='HF  3.3V  (LVPS)')
    #
    ax3.plot(hk_lvps.epoch, hk_lvps.hf_on_off,  '-k', markersize=3, label='HF ON/off (LVPS)')
    ax3.plot(hk_hf.epoch,   hk_hf.v1_8+.1,      '-r', markersize=3, label='HF 1.8V off/ON (HF)')
    #
    ax4.plot(hk_lvps.epoch, hk_lvps.cur_hf_85*10, '.g', markersize=3, label='RWI 8.5V (LVPS) [x10]')
    ax4.plot(hk_lvps.epoch, hk_lvps.cur_hf_33,  '.k', markersize=3, label='HF  3.3V (LVPS)')
if err_hf == 0:
    ax3.plot(hk_hf.epoch,   hk_hf.heater_ena,   '-y', markersize=3, label='RWI Heater ENA')
ax3.set_ylabel('Voltage [V]');  ax3.legend()
ax4.set_ylabel('Current [A]');  ax4.legend();  ax4.set_xlabel('UT')

"""
print(hk_hf.epoch)
E_min = '2024-01-25 11:00:00';  t_min = datetime.datetime.strptime(E_min, "%Y-%m-%d %H:%M:%S");  
E_max = '2024-01-25 12:10:00';  t_max = datetime.datetime.strptime(E_max, "%Y-%m-%d %H:%M:%S");  
xlim=[t_min, t_max]; ax1.set_xlim(xlim)
"""
#-----------------------------------------------------------------
fig.subplots_adjust(hspace=0)
if dump_mode == 1:
    png_fname = work_dir+'JUICE_RPWI_HF_HK_'+date+'.png'
    fig.savefig(png_fname)

In [ ]:
title_label = 'JUICE/RPWI HF HK ' + date[0:4] + '-' + date[4:6] + '-' + date[6:8]
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, figsize=(14, 11), sharex=True)

if err_hf == 0:
    ax1.plot(hk_hf.epoch, temp_hf_fpga, '.c', markersize=6, linewidth=0.5, label='HF(FPGA)')
if err_dpu == 0:
    ax1.plot(hk_dpu.epoch, hk_dpu.hf_temp,    '.k', markersize=5, label='HF(PCB)')
    ax1.plot(hk_dpu.epoch, hk_dpu.lvps_temp,  '--b', markersize=1, label='LVPS(PCB)')
ax1.set_ylabel('HF T [degC]');  ax1.legend();  ax1.set_title(title_label)

if err_hf == 0:
    ax2.plot(hk_hf.epoch, temp_rwi_ch1, '.c', markersize=6, linewidth=0.5, label='RWI CH1')
    ax2.plot(hk_hf.epoch, temp_rwi_ch2, '.y', markersize=6, linewidth=0.5, label='RWI CH2')
if err_dpu == 0:
    ax2.plot(hk_dpu.epoch, hk_dpu.scm_temp,  '.b', markersize=1, label='SCM')
ax2.set_ylabel('RWI T [degC]');  ax2.legend()

if err_hf == 0:
    ax3.plot(hk_hf.epoch,   hk_hf.rwi_on,       '-c', markersize=6, linewidth=0.5, label='RWI ON (1:U + 2:V + 4:W)')
if err_lvps == 0:
    ax3.plot(hk_lvps.epoch, hk_lvps.vol_hf_85,  '.g', markersize=3, label='RWI 8.5V (LVPS)')
    ax3.plot(hk_lvps.epoch, hk_lvps.vol_hf_33,  '.k', markersize=3, label='HF  3.3V  (LVPS)')
    #
    ax3.plot(hk_lvps.epoch, hk_lvps.hf_on_off,  '-k', markersize=3, label='HF ON/off (LVPS)')
    ax3.plot(hk_hf.epoch,   hk_hf.v1_8+.1,      '-r', markersize=3, label='HF 1.8V off/ON (HF)')
    #
    ax4.plot(hk_lvps.epoch, hk_lvps.cur_hf_85*10, '.g', markersize=3, label='RWI 8.5V (LVPS) [x10]')
    ax4.plot(hk_lvps.epoch, hk_lvps.cur_hf_33,  '.k', markersize=3, label='HF  3.3V (LVPS)')
if err_hf == 0:
    ax3.plot(hk_hf.epoch,   hk_hf.heater_ena,   '-y', markersize=3, label='RWI Heater ENA')
ax3.set_ylabel('Voltage [V]');  ax3.legend()
ax4.set_ylabel('Current [A]');  ax4.legend();  ax4.set_xlabel('UT')

print(hk_hf.epoch[0], hk_hf.epoch[-1])
t_min = hk_hf.epoch[0];  t_max = hk_hf.epoch[-1]
# E_min = '2026-02-23 00:00:00';  t_min = datetime.datetime.strptime(E_min, "%Y-%m-%d %H:%M:%S");  
# E_max = '2026-02-23 01:00:00';  t_max = datetime.datetime.strptime(E_max, "%Y-%m-%d %H:%M:%S");  
xlim=[t_min, t_max]; ax1.set_xlim(xlim)
#-----------------------------------------------------------------
fig.subplots_adjust(hspace=0)
if dump_mode == 1:
    png_fname = work_dir+'JUICE_RPWI_HF_HK_'+date+'.png'
    fig.savefig(png_fname)